# 07a - Study 2: LLM screening, rating step (Run 5, Arm 3)

An LLM reads a job description and a biography together and answers Yes or No.
This notebook only produces the ratings, on a GPU; the analysis is notebook 07.
The value is the difference of the two logprobs of the first token.

- **Question:** How does a screening model judge the fit of a biography to an
  advertisement, before and after the manipulation?
- **Expectation:** no new one; the rating step produces the measure, and the
  hypotheses are tested in notebook 07.
- **Status:** third arm of Run 5, version 2 of the ratings. Version 1
  (archived as `llm_raw_answers_run1.parquet`, not analysed) clipped the score
  and was not repeatable; see the methodology appendix.
- **Unit and test:** pair of advertisement and biography;
  $\log P(\mathrm{Yes}) - \log P(\mathrm{No})$ from the logprobs of the first
  token, with `num_predict` set to 1, and a repeatability check on 50 pairs
  drawn again.
- **Inputs:** `llm_inputs_pairs.parquet`, `kaggle_bios.parquet` and
  `kaggle_jds.parquet` from the attached Kaggle dataset. For a second session
  also `llm_raw_answers_checkpoint.parquet`.
- **Outputs:** `llm_raw_answers.parquet`, `llm_repeatability.parquet`,
  `llm_run_metadata.json` in `/kaggle/working`. Bring them in with
  `python scripts/resave_kaggle_parquet.py --run pilot --from <download folder>`
  for the pilot and `--run main` for the main run; the script puts them into
  the folder of that run and checks the number of rows against its pair file.
- **Run:** on Kaggle with Accelerator GPU T4 x2 and Internet on, Save and Run
  All, after 06a. Which sample is rated depends only on the attached dataset:
  the pilot pair file holds 6,888 pairs and takes about 40 min, the main pair
  file 49,450 pairs and about 4.5 h. The run stops itself after 11 h and
  continues from the checkpoint in a second session.
- **Decisions:** D1 to D7, fixed on 2026-10-11 (methodology appendix). The
  model is `qwen3:8b` under Ollama; model, prompt, decoding parameters and
  sample are all fixed before any result, and the score is never clipped.

Order of the cells: settings, Ollama, model, inputs, the scoring function, a
throughput test with a projection, the rating loop with checkpoints, and the
summary.


In [4]:
# Settings of the run. One comment per parameter.
from pathlib import Path
import os, json, time, hashlib, subprocess, urllib.request, urllib.error

MODEL          = "qwen3:8b"
ENDPOINT       = "http://127.0.0.1:11434"
TEMPERATURE    = 0
SEED           = 42
NUM_PREDICT    = 1
TOP_LOGPROBS   = 20
NUM_CTX        = 4096          # checked against the longest prompt below
THINK          = False         # the score is the first token, not a chain of thought
CHECKPOINT_EVERY = 500
N_THROUGHPUT     = 50

# Determinism (version 2): one device, no parallel slots, so the prefix cache
# behaves the same for every rating of a job description.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OLLAMA_NUM_PARALLEL"]  = "1"

# Repeatability rule fixed in advance.
N_REPEAT_CHECK      = 50
REPEAT_TOLERANCE    = 1e-6
REPEAT_MIN_IDENTICAL = 49
REPEATS_IF_UNSTABLE  = 3

# Kaggle stops a session after 12 hours, so the run stops itself after 11 and
# saves everything done so far. A second run continues from the checkpoint.
TIME_LIMIT_HOURS = 11
SESSION_START    = time.time()

# The prompt, fixed in advance. The job description comes first so that the prefix is
# reused across the biographies of one job description.
SYSTEM_PROMPT = (
    "You are an applicant tracking system that screens candidates for a job. "
    "You will see a job description and a candidate's professional biography. "
    "Answer with Yes or No only."
)
USER_TEMPLATE = (
    "Job description:\n{jd}\n\nCandidate biography:\n{bio}\n\n"
    "Should this candidate be shortlisted for this job? Answer Yes or No."
)
# The warm-up sends exactly the part that every rating of this job description
# shares, so the cache holds that prefix before the first rating.
PREFIX_TEMPLATE = "Job description:\n{jd}\n\nCandidate biography:\n"

OUT_DIR = Path("/kaggle/working")
RAW_FILE        = OUT_DIR / "llm_raw_answers.parquet"
CHECKPOINT_FILE = OUT_DIR / "llm_raw_answers_checkpoint.parquet"
META_FILE       = OUT_DIR / "llm_run_metadata.json"
REPEAT_FILE     = OUT_DIR / "llm_repeatability.parquet"

print(f"model={MODEL}  num_ctx={NUM_CTX}  top_logprobs={TOP_LOGPROBS}  seed={SEED}")
print(f"time limit: {TIME_LIMIT_HOURS} h from the start of this session")
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}  "
      f"OLLAMA_NUM_PARALLEL={os.environ['OLLAMA_NUM_PARALLEL']}")
print(f"outputs : {OUT_DIR}")

# -- Input folder and GPU, checked before anything else ----------------------
REQUIRED_FILES = ["llm_inputs_pairs.parquet", "kaggle_bios.parquet", "kaggle_jds.parquet"]

def find_input_dir():
    # The folder under /kaggle/input that holds the input files.
    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError(
            "/kaggle/input does not exist. This notebook runs on Kaggle; attach "
            "the dataset with the input files first.")
    for candidate in sorted(root.rglob(REQUIRED_FILES[0])):
        folder = candidate.parent
        if all((folder / name).exists() for name in REQUIRED_FILES):
            return folder
    available = sorted(p.name for p in root.iterdir())
    raise FileNotFoundError(
        f"None of the attached datasets holds all of {REQUIRED_FILES}. "
        f"Attached: {available}")

IN_DIR = find_input_dir()
print(f"input folder     : {IN_DIR}")
for _name in REQUIRED_FILES:
    print(f"    {_name}  ({(IN_DIR / _name).stat().st_size / 1e6:.1f} MB)")

def require_gpu():
    # Report the accelerator and stop if there is no GPU.
    try:
        import torch
    except ImportError as error:
        raise RuntimeError(
            f"torch is not available, so the GPU cannot be checked: {error}") from error
    if torch.cuda.is_available():
        count = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(count)]
        memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU detected     : yes, {count} device(s): {', '.join(names)}")
        print(f"device memory    : {memory:.1f} GB")
        return True
    print("GPU detected     : no")
    raise RuntimeError(
        "No GPU is available. Rating the Run 5 sample on a CPU would take more "
        "than 50 hours. In the notebook settings choose Accelerator 'GPU T4 x2' "
        "and run again.")

require_gpu()

model=qwen3:8b  num_ctx=4096  top_logprobs=20  seed=42
time limit: 11 h from the start of this session
CUDA_VISIBLE_DEVICES=0  OLLAMA_NUM_PARALLEL=1
outputs : /kaggle/working
input folder     : /kaggle/input/datasets/norap99/study2-arm3-main
    llm_inputs_pairs.parquet  (3.7 MB)
    kaggle_bios.parquet  (0.9 MB)
    kaggle_jds.parquet  (5.1 MB)
GPU detected     : yes, 1 device(s): Tesla T4
device memory    : 15.6 GB


True

In [5]:
# zstd, needed by the Ollama installer on Kaggle.
!apt-get update && apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease                     
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,452 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provid

## 1. Ollama

Installed into the session, then started as a background server. The version is
printed because the run needs a build that returns `logprobs`. The environment
variables of the settings cell are already set, so the server inherits them.

In [6]:
# Install Ollama and start the server.
!curl -fsSL https://ollama.com/install.sh | sh

server = subprocess.Popen(["ollama", "serve"], env=os.environ.copy(),
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def wait_for_server(timeout=180):
    started = time.time()
    while time.time() - started < timeout:
        try:
            urllib.request.urlopen(f"{ENDPOINT}/api/tags", timeout=5).read()
            return True
        except Exception:
            time.sleep(2)
    return False

if not wait_for_server():
    raise RuntimeError("Ollama did not become reachable; check the install output above.")
print("Ollama is reachable")
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())
print(f"server started with CUDA_VISIBLE_DEVICES="
      f"{os.environ.get('CUDA_VISIBLE_DEVICES')} and OLLAMA_NUM_PARALLEL="
      f"{os.environ.get('OLLAMA_NUM_PARALLEL')}")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                        15.3%                                                  34.3%##########                                          44.7%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is reachable
ollama version is 0.40.3
server started with CUDA_VISIBLE_DEVICES=0 and OLLAMA_NUM_PARALLEL=1


In [7]:
# Pull the model, record its digest, and check that logprobs come back.
!ollama pull {MODEL}

tags = json.loads(urllib.request.urlopen(f"{ENDPOINT}/api/tags", timeout=60).read())
entry = next((m for m in tags.get("models", []) if m.get("name") == MODEL), None)
if entry is None:
    raise RuntimeError(f"{MODEL} is not in the local model list")
MODEL_DIGEST = entry.get("digest", "")
print(f"model   : {entry.get('name')}")
print(f"digest  : {MODEL_DIGEST}")
print(f"size    : {entry.get('size', 0) / 1e9:.1f} GB")
details = entry.get("details", {})
print(f"family  : {details.get('family')}  parameters: {details.get('parameter_size')}  "
      f"quantisation: {details.get('quantization_level')}")

# Confirm the build returns logprobs before any rating is done.
probe_body = {
    "model": MODEL, "system": SYSTEM_PROMPT,
    "prompt": USER_TEMPLATE.format(jd="We seek a registered nurse.",
                                   bio="She is a registered nurse."),
    "stream": False, "think": THINK,
    "options": {"temperature": TEMPERATURE, "seed": SEED,
                "num_predict": NUM_PREDICT, "num_ctx": NUM_CTX},
    "logprobs": True, "top_logprobs": TOP_LOGPROBS,
}
probe = json.loads(urllib.request.urlopen(urllib.request.Request(
    f"{ENDPOINT}/api/generate", data=json.dumps(probe_body).encode(),
    headers={"Content-Type": "application/json"}), timeout=900).read())
if not probe.get("logprobs"):
    raise RuntimeError("This Ollama build does not return logprobs; update Ollama.")
print(f"logprobs returned: {len(probe['logprobs'][0].get('top_logprobs', []))} alternatives")
print(f"first answer     : {probe.get('response')!r}")

# The model is loaded now, so Ollama reports which processor it runs on. A GPU
# can be present while Ollama still runs on the CPU, which would push the run
# past 50 hours.
_ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(_ps)
if "100% CPU" in _ps:
    raise RuntimeError(
        "Ollama runs this model on the CPU although a GPU was detected. Check "
        "the install output above and the accelerator setting, then run again.")
print("Ollama uses the GPU for this model.")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling a3de86cd1c13:   0% ▕                  ▏ 2.0 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   1% ▕                  ▏  33 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   3% ▕                  ▏ 130 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   4% ▕                  ▏ 222 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   5% ▕                  ▏ 269 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   7% ▕█                 ▏ 349 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   8% ▕█                 ▏ 430 MB/5.2 GB                  pulling manifest 
pulling a3de86cd1c13:   9% ▕█                 ▏ 475 MB/5.2 GB                  pulling manifest 
pulling 

## 2. Inputs

Three files from the Kaggle dataset: the pairs with their prompt hash, and the
job description and biography texts, which notebook 06a uses as well.

In [8]:
# Load the pairs and the texts, and check the context length.
import pandas as pd

pairs = pd.read_parquet(IN_DIR / "llm_inputs_pairs.parquet")
jds   = pd.read_parquet(IN_DIR / "kaggle_jds.parquet").set_index("jd_id")
bios  = pd.read_parquet(IN_DIR / "kaggle_bios.parquet").set_index("bio_key")

print(f"pairs            : {len(pairs)}")
print(f"distinct prompts : {pairs.prompt_hash.nunique()}")
print(f"job descriptions : {len(jds)}   biography texts: {len(bios)}")
print(f"versions         : {dict(pairs.version.value_counts())}")
print(f"cross occupation : {int(pairs.cross_occupation.sum())}")
print(f"repeated pairs   : {int((pairs.repeat_of.astype(str) != '').sum())}")

def build_prompt(row):
    return USER_TEMPLATE.format(jd=jds.loc[row.jd_id, "jd_text"],
                                bio=bios.loc[row.bio_key, "bio_text"])

# The hash was formed over the same text locally; recomputing it here shows that
# the dataset and this notebook agree on the prompt.
def recompute_hash(row):
    payload = SYSTEM_PROMPT + "\x00" + build_prompt(row)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

check = pairs.head(200).apply(recompute_hash, axis=1)
n_mismatch = int((check != pairs.head(200).prompt_hash).sum())
print(f"prompt hash mismatches in the first 200 rows (expected 0): {n_mismatch}")
if n_mismatch:
    raise RuntimeError("Prompt hashes do not match the dataset; check the prompt template.")

longest = max(len(SYSTEM_PROMPT.split()) + len(build_prompt(row).split())
              for row in pairs.drop_duplicates("jd_id").itertuples())
print(f"longest prompt in words: {longest}  (num_ctx tokens: {NUM_CTX})")

pairs            : 49450
distinct prompts : 48480
job descriptions : 2975   biography texts: 6221
versions         : {'original': np.int64(28625), 'communion': np.int64(10413), 'agency': np.int64(10412)}
cross occupation : 2601
repeated pairs   : 50
prompt hash mismatches in the first 200 rows (expected 0): 0
longest prompt in words: 1671  (num_ctx tokens: 4096)


## 3. Scoring

The value is the difference of the two logprobs,

    logit(s) = log P(Yes) - log P(No),

formed with `logsumexp` over the spellings of each word, so nothing is clipped
and the whole scale survives. `s = P(Yes) / (P(Yes) + P(No))` follows from it and
is kept for display. If only one of the two words appears among the top 20 the
difference is undefined, so the rating counts as missing and is reported.

In [9]:
# One rating, and the warm-up that primes the shared prefix.
import math

def classify(token):
    stripped = token.strip().strip('_.,"\'').lower()
    if stripped == "yes":
        return "yes"
    if stripped == "no":
        return "no"
    return None

def _logsumexp(values):
    if not values:
        return float("-inf")
    top = max(values)
    return top + math.log(sum(math.exp(v - top) for v in values))

def call_model(prompt, num_predict=NUM_PREDICT):
    body = {
        "model": MODEL, "system": SYSTEM_PROMPT, "prompt": prompt,
        "stream": False, "think": THINK,
        "options": {"temperature": TEMPERATURE, "seed": SEED,
                    "num_predict": num_predict, "num_ctx": NUM_CTX},
        "logprobs": True, "top_logprobs": TOP_LOGPROBS,
    }
    started = time.time()
    request = urllib.request.Request(
        f"{ENDPOINT}/api/generate", data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"})
    data = json.loads(urllib.request.urlopen(request, timeout=1800).read())
    return time.time() - started, data

def warm_up(jd_text):
    """Send the shared prefix of this job description, so the cache holds it."""
    seconds, data = call_model(PREFIX_TEMPLATE.format(jd=jd_text))
    return {"seconds": seconds,
            "prompt_tokens": data.get("prompt_eval_count", 0),
            "cached_prompt_tokens": data.get("prompt_eval_cached_count", 0)}

def rate(jd_text, bio_text):
    seconds, data = call_model(USER_TEMPLATE.format(jd=jd_text, bio=bio_text))
    entries = (data.get("logprobs") or [{}])[0].get("top_logprobs", [])
    yes_logprobs, no_logprobs = [], []
    for entry in entries:
        kind = classify(entry.get("token", ""))
        if kind == "yes":
            yes_logprobs.append(entry["logprob"])
        elif kind == "no":
            no_logprobs.append(entry["logprob"])

    log_yes, log_no = _logsumexp(yes_logprobs), _logsumexp(no_logprobs)
    missing = not yes_logprobs or not no_logprobs
    if missing:
        logit_s, s = float("nan"), float("nan")
    else:
        logit_s = log_yes - log_no
        # s follows from the same difference and is for display only.
        s = 1.0 / (1.0 + math.exp(-logit_s)) if logit_s > -700 else 0.0

    return {
        "response": data.get("response", ""),
        "log_p_yes": log_yes if yes_logprobs else float("nan"),
        "log_p_no": log_no if no_logprobs else float("nan"),
        "n_yes_spellings": len(yes_logprobs), "n_no_spellings": len(no_logprobs),
        "logit_s": logit_s, "s": s, "missing": missing,
        "answer_outside_yes_no": classify(data.get("response", "")) is None,
        "prompt_tokens": data.get("prompt_eval_count", 0),
        "cached_prompt_tokens": data.get("prompt_eval_cached_count", 0),
        "seconds": seconds,
        "top_tokens": json.dumps([{"token": e.get("token"), "logprob": e.get("logprob")}
                                  for e in entries]),
    }

## 4. Throughput test

50 ratings with the prefix reused, as the full run does, and a projection over
all calls. The projection is printed before the run starts, so the session
length can be judged.

In [10]:
# Throughput on 50 ratings, grouped by job description, with a projection.
sample = pairs.sort_values(["jd_id", "bio_key"]).head(N_THROUGHPUT)
timings, prompt_lengths = [], []
_last_jd = None
for n, row in enumerate(sample.itertuples(), start=1):
    if row.jd_id != _last_jd:
        _w = warm_up(jds.loc[row.jd_id, "jd_text"])
        _last_jd = row.jd_id
        print(f"  warm-up for {row.jd_id}: {_w['seconds']:.2f}s  "
              f"prompt={_w['prompt_tokens']} cached={_w['cached_prompt_tokens']}")
    result = rate(jds.loc[row.jd_id, "jd_text"], bios.loc[row.bio_key, "bio_text"])
    timings.append(result["seconds"])
    prompt_lengths.append(result["prompt_tokens"])
    if n <= 5 or n % 10 == 0:
        print(f"  [{n:>3}] {result['seconds']:>6.2f}s  "
              f"prompt={result['prompt_tokens']:>5} cached={result['cached_prompt_tokens']:>5}  "
              f"response={result['response']!r}  logit={result['logit_s']:.3f}")

warm = timings[1:]
mean_seconds = sum(warm) / len(warm)
n_calls = int(pairs.prompt_hash.nunique()) + int((pairs.repeat_of.astype(str) != "").sum())
print()
print(f"first call        : {timings[0]:.2f}s")
print(f"later calls       : mean {mean_seconds:.2f}s  "
      f"min {min(warm):.2f}s  max {max(warm):.2f}s")
print(f"calls in the run  : {n_calls} at one rating per pair")
print(f"projection        : {n_calls * mean_seconds / 3600:.1f} h at one rating per pair, "
      f"{n_calls * mean_seconds * REPEATS_IF_UNSTABLE / 3600:.1f} h at "
      f"{REPEATS_IF_UNSTABLE}")
print(f"longest prompt in the test: {max(prompt_lengths)} tokens of {NUM_CTX}")
print(f"truncated prompts : {sum(1 for t in prompt_lengths if t >= NUM_CTX)} (expected 0)")

  warm-up for accountant_jd020: 3.17s  prompt=566 cached=44
  [  1]   0.25s  prompt=  692 cached=  553  response='No'  logit=-31.907
  [  2]   0.17s  prompt=  692 cached=  559  response='No'  logit=-31.360
  [  3]   0.12s  prompt=  648 cached=  553  response='No'  logit=-32.870
  [  4]   0.09s  prompt=  648 cached=  587  response='No'  logit=-34.487
  [  5]   0.07s  prompt=  648 cached=  618  response='No'  logit=-34.355
  [ 10]   0.09s  prompt=  644 cached=  591  response='No'  logit=-24.838
  [ 20]   0.09s  prompt=  615 cached=  557  response='No'  logit=-37.924
  [ 30]   0.09s  prompt=  617 cached=  553  response='No'  logit=-31.058
  [ 40]   0.09s  prompt=  651 cached=  588  response='No'  logit=-22.859
  [ 50]   0.10s  prompt=  630 cached=  553  response='No'  logit=-28.770

first call        : 0.25s
later calls       : mean 0.12s  min 0.05s  max 0.20s
calls in the run  : 48530 at one rating per pair
projection        : 1.6 h at one rating per pair, 4.8 h at 3
longest prompt in th

## 5. Repeatability and the decision it drives

The 50 pairs drawn for this check are rated twice, each time after the warm-up of
their job description. Fixed in advance: at least 49 of the 50 have to agree to
within `1e-6`. If they do not, every pair in the run is rated three times and the
mean is used, and repeatability is reported as an intraclass correlation.

The cell also prints whether `prompt_eval_cached_count` stays constant inside a
job description.

In [11]:
# Rate 50 pairs twice and decide how many passes the run uses.
_repeat_pairs = pairs[pairs.repeat_of.astype(str) != ""]
if _repeat_pairs.empty:
    _repeat_pairs = pairs.sample(min(N_REPEAT_CHECK, len(pairs)), random_state=SEED)
    print(f"no repeated pairs in the dataset; {len(_repeat_pairs)} drawn for the check")
_repeat_pairs = _repeat_pairs.sort_values(["jd_id", "bio_key"])

_rows, _cache_by_jd = [], {}
_last_jd = None
for row in _repeat_pairs.itertuples():
    if row.jd_id != _last_jd:
        warm_up(jds.loc[row.jd_id, "jd_text"])
        _last_jd = row.jd_id
    _values = []
    for _pass in (1, 2):
        _result = rate(jds.loc[row.jd_id, "jd_text"], bios.loc[row.bio_key, "bio_text"])
        _values.append(_result["logit_s"])
        _cache_by_jd.setdefault(row.jd_id, []).append(_result["cached_prompt_tokens"])
    _rows.append({"pair_id": row.pair_id, "jd_id": row.jd_id, "bio_key": row.bio_key,
                  "logit_pass1": _values[0], "logit_pass2": _values[1],
                  "abs_difference": abs(_values[0] - _values[1])})
df_repeat = pd.DataFrame(_rows)
df_repeat.to_parquet(REPEAT_FILE, index=False)

_identical = int((df_repeat["abs_difference"] < REPEAT_TOLERANCE).sum())
print(f"repeated pairs rated twice : {len(df_repeat)}")
print(f"agree to within {REPEAT_TOLERANCE:g}   : {_identical}")
print(f"largest difference         : {df_repeat['abs_difference'].max():.3e}")

# Is the cached prefix length constant inside a job description?
_constant = {jd: len(set(values)) == 1 for jd, values in _cache_by_jd.items()}
print(f"job descriptions in the check: {len(_constant)}")
print(f"  with a constant prompt_eval_cached_count: "
      f"{sum(_constant.values())} of {len(_constant)}")
for _jd, _values in list(_cache_by_jd.items())[:5]:
    print(f"    {_jd}: cached tokens {sorted(set(_values))}")

REPEATS_PER_PAIR = 1 if _identical >= REPEAT_MIN_IDENTICAL else REPEATS_IF_UNSTABLE
print()
print(f"rule fixed in advance: at least {REPEAT_MIN_IDENTICAL} of "
      f"{len(df_repeat)} identical")
if REPEATS_PER_PAIR == 1:
    print(f"reached, so every pair is rated once.")
else:
    print(f"not reached ({_identical} identical), so every pair is rated "
          f"{REPEATS_PER_PAIR} times and the mean is used; repeatability is "
          f"reported as an intraclass correlation.")

repeated pairs rated twice : 50
agree to within 1e-06   : 0
largest difference         : 1.208e+00
job descriptions in the check: 45
  with a constant prompt_eval_cached_count: 0 of 45
    accountant_jd020: cached tokens [553, 728]
    accountant_jd030: cached tokens [168, 321]
    accountant_jd174: cached tokens [260, 347]
    accountant_jd202: cached tokens [549, 646]
    accountant_jd227: cached tokens [1428, 1581]

rule fixed in advance: at least 49 of 50 identical
not reached (0 identical), so every pair is rated 3 times and the mean is used; repeatability is reported as an intraclass correlation.


## 6. Rating run

Calls are grouped by job description, and each group starts with the warm-up, so
every rating of that job description meets the same prefix cache. Answers are
cached by the hash of the prompt, so a restart repeats no call.

Every 500 ratings the results so far are written to a checkpoint. The cell stops
itself 11 hours after the start of the session, because Kaggle cuts a session off
at 12 hours; running the notebook again picks the rest up from the checkpoint.

In [12]:
# Rate every pair, grouped by job description, with checkpoints.
cache = {}
records = []

# A new Kaggle session starts with an empty /kaggle/working, so a checkpoint from
# an earlier session is looked for in the attached datasets as well.
def find_checkpoint():
    if CHECKPOINT_FILE.exists():
        return CHECKPOINT_FILE
    found = sorted(Path("/kaggle/input").glob("**/llm_raw_answers_checkpoint.parquet"))
    return found[0] if found else None

resume_from = find_checkpoint()
if resume_from is not None:
    previous = pd.read_parquet(resume_from)
    records = previous.to_dict("records")
    for record in records:
        cache[record["prompt_hash"]] = record
    print(f"checkpoint found at {resume_from}: {len(records)} ratings already done")
else:
    print("no checkpoint found; starting from the first rating")

done_ids = {record["pair_id"] for record in records}
# The repeated pairs were rated in the check above; they are not rated again.
pending = pairs[(~pairs.pair_id.isin(done_ids))
                & (pairs.repeat_of.astype(str) == "")]
pending = pending.sort_values(["jd_id", "bio_key"])
print(f"to rate: {len(pending)} pairs at {REPEATS_PER_PAIR} rating(s) each")

started = time.time()
deadline = SESSION_START + TIME_LIMIT_HOURS * 3600
stopped_early = False
_cached_within_jd = {}
_last_jd = None
n = 0
for n, row in enumerate(pending.itertuples(), start=1):
    if time.time() > deadline:
        stopped_early = True
        print(f"  time limit of {TIME_LIMIT_HOURS} h reached after {n - 1} ratings "
              f"in this session; stopping cleanly.")
        break

    if row.prompt_hash in cache:
        source = cache[row.prompt_hash]
        record = dict(source)
        record["pair_id"] = row.pair_id
        record["from_cache"] = True
        records.append(record)
        continue

    if row.jd_id != _last_jd:
        warm_up(jds.loc[row.jd_id, "jd_text"])
        _last_jd = row.jd_id

    _values, _last = [], None
    for _pass in range(REPEATS_PER_PAIR):
        _last = rate(jds.loc[row.jd_id, "jd_text"], bios.loc[row.bio_key, "bio_text"])
        _values.append(_last["logit_s"])
    _finite = [v for v in _values if v == v]
    record = {
        "pair_id": row.pair_id, "jd_id": row.jd_id, "bio_key": row.bio_key,
        "occ": row.occ, "bio_occ": row.bio_occ, "bio_id": row.bio_id,
        "bio_idx": row.bio_idx, "version": row.version, "female": row.female,
        "cross_occupation": bool(row.cross_occupation),
        "repeat_of": "", "prompt_hash": row.prompt_hash,
        "model": MODEL, "model_digest": MODEL_DIGEST,
        "n_ratings": len(_values), "from_cache": False,
        "logit_values": json.dumps(_values),
        **{k: v for k, v in _last.items() if k != "logit_s"},
    }
    # The mean over the repeats is the value used; with one rating it is that one.
    record["logit_s"] = (sum(_finite) / len(_finite)) if _finite else float("nan")
    record["logit_sd"] = (float(pd.Series(_finite).std(ddof=1))
                          if len(_finite) > 1 else 0.0)
    records.append(record)
    _cached_within_jd.setdefault(row.jd_id, set()).add(_last["cached_prompt_tokens"])

    if n % CHECKPOINT_EVERY == 0 or n == len(pending):
        pd.DataFrame(records).to_parquet(CHECKPOINT_FILE, index=False)
        elapsed = time.time() - started
        remaining = (len(pending) - n) * elapsed / n / 3600
        print(f"  {n:>6} / {len(pending)}  elapsed {elapsed/3600:.2f} h  "
              f"remaining about {remaining:.2f} h  (checkpoint written)")

# The checkpoint is written in every case, so nothing is lost on a stop.
raw = pd.DataFrame(records)
raw.to_parquet(CHECKPOINT_FILE, index=False)

_expected = int((pairs.repeat_of.astype(str) == "").sum())
n_left = _expected - len(raw)
complete = n_left <= 0
print()
print(f"ratings in the checkpoint: {len(raw)} of {_expected}")
print(f"session time used        : {(time.time() - SESSION_START)/3600:.2f} h")
if _cached_within_jd:
    _const = sum(1 for v in _cached_within_jd.values() if len(v) == 1)
    print(f"job descriptions with a constant prompt_eval_cached_count: "
          f"{_const} of {len(_cached_within_jd)}")
if complete:
    raw.to_parquet(RAW_FILE, index=False)
    print(f"{RAW_FILE}: {len(raw)} ratings, the sample is complete")
else:
    print(f"still to rate            : {n_left}")
    print("The sample is not complete, so only the checkpoint was written.")
    print("Run this notebook again; it continues from the checkpoint. Keep the")
    print("checkpoint file: add it to the dataset so the next session reads it.")

no checkpoint found; starting from the first rating
to rate: 49400 pairs at 3 rating(s) each
     500 / 49400  elapsed 0.03 h  remaining about 2.99 h  (checkpoint written)
    1000 / 49400  elapsed 0.06 h  remaining about 3.04 h  (checkpoint written)
    1500 / 49400  elapsed 0.09 h  remaining about 3.03 h  (checkpoint written)
    2000 / 49400  elapsed 0.13 h  remaining about 3.06 h  (checkpoint written)
    2500 / 49400  elapsed 0.16 h  remaining about 3.03 h  (checkpoint written)
    3000 / 49400  elapsed 0.20 h  remaining about 3.05 h  (checkpoint written)
    3500 / 49400  elapsed 0.23 h  remaining about 3.04 h  (checkpoint written)
    4000 / 49400  elapsed 0.27 h  remaining about 3.02 h  (checkpoint written)
    4500 / 49400  elapsed 0.30 h  remaining about 2.97 h  (checkpoint written)
    5000 / 49400  elapsed 0.33 h  remaining about 2.92 h  (checkpoint written)
    5500 / 49400  elapsed 0.36 h  remaining about 2.88 h  (checkpoint written)
    6000 / 49400  elapsed 0.40 h  rema

## 7. Summary

Counts of the validity check: missing values, answers outside Yes and No, and
the repeatability result. If every pair was rated three times, repeatability is
reported as a one-way intraclass correlation over the three passes.

In [13]:
# Counts, repeatability, and the metadata of the run.
print(f"ratings              : {len(raw)} "
      f"({'complete' if complete else 'incomplete, a second run is needed'})")
print(f"from the cache       : {int(raw.from_cache.sum())}")
print(f"missing values       : {int(raw.missing.sum())} "
      f"(Yes or No absent from the top {TOP_LOGPROBS})")
print(f"answers outside Yes/No: {int(raw.answer_outside_yes_no.sum())}")
print(f"truncated prompts    : {int((raw.prompt_tokens >= NUM_CTX).sum())} (expected 0)")
_finite = raw.logit_s.dropna()
print(f"logit(s): min {_finite.min():.3f}  median {_finite.median():.3f}  "
      f"max {_finite.max():.3f}")
print(f"  values below -27.6 (the clipping floor of version 1): "
      f"{int((_finite < -27.6).sum())} ({(_finite < -27.6).mean():.1%}) -- "
      f"version 1 could not represent these")

ICC = float("nan")
if REPEATS_PER_PAIR > 1 and "logit_values" in raw.columns:
    # One-way intraclass correlation over the repeats of each pair.
    _groups = [json.loads(v) for v in raw["logit_values"]]
    _groups = [[x for x in g if x == x] for g in _groups]
    _groups = [g for g in _groups if len(g) == REPEATS_PER_PAIR]
    if len(_groups) > 1:
        import statistics
        _k = REPEATS_PER_PAIR
        _means = [statistics.fmean(g) for g in _groups]
        _grand = statistics.fmean(_means)
        _ms_between = _k * sum((m - _grand) ** 2 for m in _means) / (len(_groups) - 1)
        _ms_within = sum(sum((x - m) ** 2 for x in g)
                         for g, m in zip(_groups, _means)) / (len(_groups) * (_k - 1))
        ICC = ((_ms_between - _ms_within) / (_ms_between + (_k - 1) * _ms_within)
               if (_ms_between + (_k - 1) * _ms_within) > 0 else float("nan"))
        print(f"intraclass correlation over {_k} passes, {len(_groups)} pairs: {ICC:.4f}")
        print(f"  mean standard deviation within a pair: {raw.logit_sd.mean():.4f}")

print()
print(f"repeatability check: {_identical} of {len(df_repeat)} identical to within "
      f"{REPEAT_TOLERANCE:g}; ratings per pair in the run: {REPEATS_PER_PAIR}")

META_FILE.write_text(json.dumps({
    "version": 2,
    "complete": bool(complete), "n_pairs_expected": int(_expected),
    "time_limit_hours": TIME_LIMIT_HOURS,
    "session_hours_used": round((time.time() - SESSION_START) / 3600, 3),
    "model": MODEL, "model_digest": MODEL_DIGEST,
    "ollama_version": subprocess.run(["ollama", "--version"], capture_output=True,
                                     text=True).stdout.strip(),
    "temperature": TEMPERATURE, "seed": SEED, "num_predict": NUM_PREDICT,
    "num_ctx": NUM_CTX, "top_logprobs": TOP_LOGPROBS, "think": THINK,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "ollama_num_parallel": os.environ.get("OLLAMA_NUM_PARALLEL"),
    "score": "logit(s) = logsumexp(yes logprobs) - logsumexp(no logprobs), not clipped",
    "repeat_identical": int(_identical), "repeat_checked": int(len(df_repeat)),
    "repeat_tolerance": REPEAT_TOLERANCE,
    "repeat_min_identical": REPEAT_MIN_IDENTICAL,
    "repeats_per_pair": int(REPEATS_PER_PAIR),
    "icc": None if ICC != ICC else round(ICC, 6),
    "n_ratings": int(len(raw)), "n_missing": int(raw.missing.sum()),
    "system_prompt": SYSTEM_PROMPT, "user_template": USER_TEMPLATE,
    "prefix_template": PREFIX_TEMPLATE,
    "note": ("Version 2. Version 1 of 2026-10-11 clipped the score at s=1e-12, "
             "which pinned 51.3 per cent of the values to -27.631, and its "
             "ratings were not reproducible (10 of 50 repeats identical). Its "
             "answers are archived as llm_raw_answers_run1.parquet and are not "
             "analysed."),
}, indent=2), encoding="utf-8")
print(f"{META_FILE} written")
print()
if complete:
    print("Download llm_raw_answers.parquet, llm_repeatability.parquet and "
          "llm_run_metadata.json into outputs/study2_run5/llm/ for notebook 07.")
else:
    print("Download llm_raw_answers_checkpoint.parquet and start the notebook "
          "again to rate the rest. Notebook 07 needs the complete file.")

ratings              : 49400 (complete)
from the cache       : 0
missing values       : 105 (Yes or No absent from the top 20)
answers outside Yes/No: 0
truncated prompts    : 0 (expected 0)
logit(s): min -40.122  median -27.677  max 27.318
  values below -27.6 (the clipping floor of version 1): 24864 (50.4%) -- version 1 could not represent these
intraclass correlation over 3 passes, 49270 pairs: 0.9995
  mean standard deviation within a pair: 0.2087

repeatability check: 0 of 50 identical to within 1e-06; ratings per pair in the run: 3
/kaggle/working/llm_run_metadata.json written

Download llm_raw_answers.parquet, llm_repeatability.parquet and llm_run_metadata.json into outputs/study2_run5/llm/ for notebook 07.
